In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-04-08 11:44:18.702346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744109058.714013 3171336 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744109058.717487 3171336 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-08 11:44:18.730655: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1744109060.096679 3171336 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14957 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-4:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-6e-6))

In [3]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnuSer'] = np.log10(df_full['dnuSer']*135)

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['lognumax', 'logdnuSer'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'lognumax':0.0001, 'logdnuSer':0.0001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [5]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 5000

learning_rate = 0.001

model_name = 'smart-logLPhot-lognumax-logdnuSer-exponent-6e-6'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [6]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/5000


I0000 00:00:1744023316.037066 2530125 service.cc:148] XLA service 0x7e956800eac0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744023316.037094 2530125 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-04-07 11:55:16.058232: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1744023316.176200 2530125 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-07 11:55:16.218265: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-07 11:55:16.75234

 61/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1855107.8750

I0000 00:00:1744023318.078618 2530125 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


402/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1471841.2500

2025-04-07 11:55:20.064758: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-04-07 11:55:20.273834: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 136 bytes spill stores, 136 bytes spill loads

2025-04-07 11:55:20.329438: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 284 bytes spill stores, 284 bytes spill loads

2025-04-07 11:55:20.366008: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 24 bytes spill stores, 24 bytes spill loads

2025-04-07 11:55:20.375068: I external/local_xla/xla/stream_

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1468962.6250

2025-04-07 11:55:22.389661: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-04-07 11:55:22.455445: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30_0', 168 bytes spill stores, 168 bytes spill loads




Epoch 1: val_loss improved from inf to 1310887.37500, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-exponent-6e-6-nlayers-6-nunits-128-epochs-5000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 1468707.2500 - val_loss: 1310887.3750 - learning_rate: 9.9999e-04
Epoch 2/5000
406/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1309991.6250
Epoch 2: val_loss did not improve from 1310887.37500
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1310116.1250 - val_loss: 1352693.3750 - learning_rate: 9.9999e-04
Epoch 3/5000
400/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1303010.6250
Epoch 3: val_loss improved from 1310887.37500 to 1264254.87500, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-exponent-6e-6-nlayers-6-nunits-128-epochs-5000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1302988.8750 - val_loss: 1264254.8

In [10]:
unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'lognumax':0.0001, 'logdnuSer':0.0001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [11]:
custom_objects =  {'WMSE':WMSE_metric}

model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)


In [12]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nepochs = 5000 +Nepochs

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=Nepochs)


Epoch 5001/10000


I0000 00:00:1744109211.576722 3171590 service.cc:148] XLA service 0x735a1c0020f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744109211.576754 3171590 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-04-08 11:46:51.617560: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1744109211.741785 3171590 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-08 11:46:51.772288: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-08 11:46:52.40745

 59/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 503332.6250

I0000 00:00:1744109213.673524 3171590 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


399/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 153248.4219

2025-04-08 11:46:55.630713: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-04-08 11:46:55.804143: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 24 bytes spill stores, 24 bytes spill loads

2025-04-08 11:46:55.978974: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 136 bytes spill stores, 136 bytes spill loads

2025-04-08 11:46:55.996497: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451_0', 36 bytes spill stores, 36 bytes spill loads

2025-04-08 11:46:56.023307: I external/local_xla/xla/stream_

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 149541.1875

2025-04-08 11:46:58.184419: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-04-08 11:46:58.250936: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30_0', 168 bytes spill stores, 168 bytes spill loads




Epoch 5001: val_loss improved from inf to 10060.73340, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-exponent-6e-6-nlayers-6-nunits-128-epochs-5000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 149284.7969 - val_loss: 10060.7334 - learning_rate: 9.4187e-04
Epoch 5002/10000
405/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 9441.9023
Epoch 5002: val_loss did not improve from 10060.73340
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 9447.4062 - val_loss: 22005.1777 - learning_rate: 9.4186e-04
Epoch 5003/10000
409/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 10480.6426
Epoch 5003: val_loss improved from 10060.73340 to 4446.34326, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-lognumax-logdnuSer-exponent-6e-6-nlayers-6-nunits-128-epochs-5000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10480.0166 - val_loss: 4446.3433 - lear

In [ ]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nepochs = 5000 +Nepochs

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=Nepochs)